# 03 - Serving speed: merge the adapter + benchmark vLLM (no retraining)
Runtime -> **T4 GPU**. Same Drive folder as 01/02. Needs `outputs/qwen3-4b-triage-lora/` from notebook 02.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/finetuned_llm
!nvidia-smi | head -15

## 1. Merge the LoRA adapter into the base model (~3-5 min)

In [ ]:
%%capture
!pip install -q transformers accelerate peft scikit-learn

In [ ]:
!python scripts/merge_adapter.py

## 2. Install vLLM (~5 min). If Colab asks to restart the session, restart and continue from the next cell (do not re-run installs).

In [ ]:
%%capture
!pip install -q vllm

## 3. Benchmark: 25 real tickets + 100 synthetic, one at a time, 5 warm-up requests excluded

In [ ]:
!python scripts/bench_vllm.py --name qwen-ft-vllm --splits real_test test --limits 25 100

## 4. Score accuracy + latency of the merged/vLLM model

In [ ]:
!python scripts/report.py --errors

## 5. Download the two prediction files

In [ ]:
from google.colab import files
files.download('outputs/preds_qwen-ft-vllm_real_test.jsonl')
files.download('outputs/preds_qwen-ft-vllm_test.jsonl')